In [2]:
import torch 
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

In [ ]:
class LstmClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, bidirectional = True):
        super().__init__()

        self.input_sz = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers,bidirectional=bidirectional)
        self.clf = nn.Sequential(
            nn.Linear(2 * hidden_size, 4 * hidden_size),
            nn.LayerNorm(4 * hidden_size),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(4 * hidden_size, 1)
        )

    def forward(self, x, lengths):
        B, T, F = x.shape
        packed_x = pack_padded_sequence(x, lengths=lengths.cpu(), batch_first=True, enforce_sorted=False)

        output, (h, c) = self.lstm(packed_x) # output -> T, B, 2*H, h -> B, 2*H, c -> B, 2*H
        clf_tensor = torch.cat([h[-2], h[-1]], dim=-1)
        logits = self.clf(clf_tensor)

        return logits

